# CNN Training Experiments

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f" GPU : {len(gpus)} ")
    for g in gpus:
        print(f"   {g}")
else:
    print(" GPU tidak tersedia")

print(f"TensorFlow: {tf.__version__}")
print(f"CUDA: {tf.test.is_built_with_cuda()}")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/CNN_Experiments'
MODEL_DIR  = os.path.join(DRIVE_BASE, 'models')
RESULT_DIR = os.path.join(DRIVE_BASE, 'results')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print(f"Model dir  : {MODEL_DIR}")
print(f"Result dir : {RESULT_DIR}")

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import json, itertools, os
from pathlib import Path
from datetime import datetime

In [ ]:
import kagglehub

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
except Exception:
    print("Belum ada KAGGLE_USERNAME & KAGGLE_KEY")

DATASET_PATH = Path(kagglehub.dataset_download("puneet6060/intel-image-classification"))
print(f"\nDataset: {DATASET_PATH}")
for p in sorted(DATASET_PATH.iterdir()):
    print(f"  {p.name}")

In [ ]:
IMG_SIZE    = (150, 150)
CHANNELS    = 3
RANDOM_SEED = 42

train_dir = DATASET_PATH / 'seg_train' / 'seg_train'
test_dir  = DATASET_PATH / 'seg_test'  / 'seg_test'

for d, label in [(train_dir, 'train'), (test_dir, 'test')]:
    if d.exists():
        classes = [x.name for x in sorted(d.iterdir()) if x.is_dir()]
        print(f"{label:5s}: {d}")
        print(f"   classes: {classes}")
    else:
        print(f"{label} dir tidak ditemukan: {d}")

In [ ]:
from PIL import Image

def image_loader(path, h=128, w=128):
    im = Image.open(path)
    if h != None and w != None:
        im = im.resize((w, h))
    im_arr = np.array(im)
    norm_im = (im_arr  - np.min(im_arr))/(np.max(im_arr) - np.min(im_arr))
    return norm_im

def batch_loader(paths, h=128, w=128, c=3):
    batch_array = np.zeros((len(paths), h, w, c), dtype=np.float32)    
    for idx, path in enumerate(paths):
        img = image_loader(path, h, w)
        if c == 1 and len(img.shape) == 2:
            img = np.expand_dims(img, axis=-1)
        elif c == 3 and len(img.shape) == 2:
            img = np.stack([img, img, img], axis=-1)
        batch_array[idx] = img[:h, :w, :c]
    return batch_array

def collect_paths_and_labels(directory):
    directory = Path(directory)
    class_folders = sorted([d for d in directory.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_folders]

    all_paths, all_labels = [], []
    for class_idx, class_folder in enumerate(class_folders):
        image_paths = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            image_paths.extend(class_folder.glob(ext))
            image_paths.extend(class_folder.glob(ext.upper()))
        image_paths = sorted([str(p) for p in image_paths])
        print(f"  {class_folder.name}: {len(image_paths)} images")
        all_paths.extend(image_paths)
        all_labels.extend([class_idx] * len(image_paths))

    return all_paths, np.array(all_labels, dtype=np.int32), class_names

def make_tf_dataset(paths, labels, img_size=(128,128), batch_size=32, shuffle=False):
    CHUNK = 64

    def load_image_tf(path, label):
        img = tf.numpy_function(
            func=lambda p: image_loader(p.decode(), h=img_size[0], w=img_size[1]),
            inp=[path],
            Tout=tf.float32
        )
        img.set_shape([img_size[0], img_size[1], 3])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=2000, seed=RANDOM_SEED)
    ds = ds.map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_paths, y_train_full, class_names = collect_paths_and_labels(train_dir)
test_paths, y_test, _ = collect_paths_and_labels(test_dir)
num_classes = len(class_names)
print(f"\n Train: {len(train_paths)} | Test: {len(test_paths)} | Classes: {class_names}")

In [ ]:
from sklearn.model_selection import train_test_split

train_paths = np.array(train_paths)

X_train_paths, X_val_paths, y_train, y_val = train_test_split(
    train_paths, y_train_full,
    test_size=0.2, random_state=RANDOM_SEED, stratify=y_train_full
)

print(f"Train      : {len(X_train_paths)} images")
print(f"Validation : {len(X_val_paths)} images")
print(f"Test       : {len(test_paths)} images")

Train      : 11227 images
Validation : 2807 images
Test       : 3000 images


In [ ]:
HYPERPARAMS = {
    'num_conv_layers': [2, 3],
    'filters'        : [[32, 64], [64, 128]],
    'kernel_size'    : [(3, 3), (5, 5)],
    'pooling_type'   : ['max', 'average']
}

experiments = [
    {'num_conv_layers': nl, 'filters': f, 'kernel_size': k, 'pooling_type': p}
    for nl, f, k, p in itertools.product(
        HYPERPARAMS['num_conv_layers'],
        HYPERPARAMS['filters'],
        HYPERPARAMS['kernel_size'],
        HYPERPARAMS['pooling_type']
    )
]

print(f"Total experiments: {len(experiments)}")
for i, exp in enumerate(experiments):
    print(f"  {i+1:2d}. {exp}")

Total experiments: 16
   1. {'num_conv_layers': 2, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'max'}
   2. {'num_conv_layers': 2, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'average'}
   3. {'num_conv_layers': 2, 'filters': [32, 64], 'kernel_size': (5, 5), 'pooling_type': 'max'}
   4. {'num_conv_layers': 2, 'filters': [32, 64], 'kernel_size': (5, 5), 'pooling_type': 'average'}
   5. {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (3, 3), 'pooling_type': 'max'}
   6. {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (3, 3), 'pooling_type': 'average'}
   7. {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'max'}
   8. {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'average'}
   9. {'num_conv_layers': 3, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'max'}
  10. {'num_conv_layers': 3, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'aver

In [ ]:
def build_cnn_model(input_shape, num_classes, config):
    model = keras.Sequential(name=f"cnn_{config['num_conv_layers']}layers")
    model.add(layers.Input(shape=input_shape))

    for i in range(config['num_conv_layers']):
        nf = config['filters'][min(i, len(config['filters']) - 1)]
        model.add(layers.Conv2D(nf, config['kernel_size'],
                                padding='same', activation='relu',
                                name=f'conv_{i+1}'))
        if config['pooling_type'] == 'max':
            model.add(layers.MaxPooling2D((2, 2), name=f'maxpool_{i+1}'))
        else:
            model.add(layers.AveragePooling2D((2, 2), name=f'avgpool_{i+1}'))

    model.add(layers.GlobalAveragePooling2D(name='global_pool'))
    model.add(layers.Dense(128, activation='relu', name='dense_1'))
    model.add(layers.Dropout(0.5, name='dropout'))
    model.add(layers.Dense(num_classes, activation='softmax', name='output'))
    return model

In [ ]:
BATCH_SIZE    = 64
EPOCHS        = 10
LEARNING_RATE = 0.001

print(f"Batch size    : {BATCH_SIZE}")
print(f"Max epochs    : {EPOCHS}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Model dir     : {MODEL_DIR}")
print(f"Result dir    : {RESULT_DIR}")

Batch size    : 64
Max epochs    : 10
Learning rate : 0.001
Model dir     : /content/drive/MyDrive/CNN_Experiments/models
Result dir    : /content/drive/MyDrive/CNN_Experiments/results


In [ ]:
# Setiap model disimpan dalam DUA format:
#   .keras  — format modern TF/Keras
#   .h5     — format legacy, untuk kompatibilitas

results = []
checkpoint_json = os.path.join(RESULT_DIR, 'results_checkpoint.json')

if os.path.exists(checkpoint_json):
    with open(checkpoint_json) as f:
        results = json.load(f)
    done_ids = {r['experiment_id'] for r in results}
    print(f"↩️  Lanjut dari checkpoint: {len(results)} eksperimen selesai")
else:
    done_ids = set()

for idx, config in enumerate(experiments):
    exp_id = idx + 1
    if exp_id in done_ids:
        print(f"⏭️  Exp {exp_id} sudah selesai, skip.")
        continue

    print(f"\n{'='*80}")
    print(f"Experiment {exp_id}/{len(experiments)}  |  {config}")
    print(f"{'='*80}\n")

    model_name      = f"model_{exp_id:02d}"
    model_keras_path = os.path.join(MODEL_DIR, f"{model_name}.keras")
    model_h5_path    = os.path.join(MODEL_DIR, f"{model_name}.h5")

    model = build_cnn_model((*IMG_SIZE, CHANNELS), num_classes, config)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    if exp_id == 1:
        model.summary()

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=5, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
        keras.callbacks.ModelCheckpoint(
            filepath=model_keras_path,
            monitor='val_loss', save_best_only=True,
            verbose=0),
    ]

    start_time = datetime.now()
    train_ds = make_tf_dataset(X_train_paths, y_train,
                              img_size=IMG_SIZE, batch_size=BATCH_SIZE, shuffle=True)
    val_ds   = make_tf_dataset(X_val_paths, y_val,
                              img_size=IMG_SIZE, batch_size=BATCH_SIZE)

    history = model.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=callbacks, verbose=1
    )
    training_time = (datetime.now() - start_time).total_seconds()

    # Evaluasi
    test_ds = make_tf_dataset(test_paths, y_test,
                              img_size=IMG_SIZE, batch_size=BATCH_SIZE)

    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    y_pred_classes = np.argmax(model.predict(test_ds, verbose=0), axis=1)

    f1_macro       = f1_score(y_test, y_pred_classes, average='macro')

    # Simpan .h5
    model.save(model_h5_path)

    # Simpan history eksperimen
    history_path = os.path.join(RESULT_DIR, f"{model_name}_history.json")
    with open(history_path, 'w') as f:
        json.dump({k: [float(v) for v in vals]
                   for k, vals in history.history.items()}, f, indent=2)

    result = {
        'experiment_id'   : exp_id,
        'model_name'      : model_name,
        'config'          : {
            'num_conv_layers': config['num_conv_layers'],
            'filters'        : config['filters'],
            'kernel_size'    : list(config['kernel_size']),
            'pooling_type'   : config['pooling_type'],
        },
        'test_accuracy'   : float(test_acc),
        'test_loss'       : float(test_loss),
        'f1_macro'        : float(f1_macro),
        'training_time_sec': training_time,
        'num_params'      : int(model.count_params()),
        'best_epoch'      : len(history.history['loss']),
        'best_val_loss'   : float(min(history.history['val_loss'])),
        'best_val_acc'    : float(max(history.history['val_accuracy'])),
    }
    results.append(result)

    # Checkpoint
    with open(checkpoint_json, 'w') as f:
        json.dump(results, f, indent=2)

    print(f"\n✅ Exp {exp_id}: acc={test_acc:.4f} | loss={test_loss:.4f} | "
          f"F1={f1_macro:.4f} | {training_time:.1f}s")
    print(f"   💾 {model_keras_path}")
    print(f"   💾 {model_h5_path}")
    print(f"   💾 {history_path}")

    keras.backend.clear_session()

print(f"\n{'='*80}")
print("✅ Semua eksperimen selesai!")
print(f"{'='*80}")

↩️  Lanjut dari checkpoint: 6 eksperimen selesai
⏭️  Exp 1 sudah selesai, skip.
⏭️  Exp 2 sudah selesai, skip.
⏭️  Exp 3 sudah selesai, skip.
⏭️  Exp 4 sudah selesai, skip.
⏭️  Exp 5 sudah selesai, skip.
⏭️  Exp 6 sudah selesai, skip.

Experiment 7/16  |  {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'max'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 30s 148ms/step - accuracy: 0.3987 - loss: 1.4059 - val_accuracy: 0.5137 - val_loss: 1.1788 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 24s 135ms/step - accuracy: 0.4897 - loss: 1.1765 - val_accuracy: 0.5479 - val_loss: 1.0855 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 42s 140ms/step - accuracy: 0.5356 - loss: 1.0999 - val_accuracy: 0.5757 - val_loss: 1.0476 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 41s 139ms/step - accuracy: 0.5658 - loss: 1.0526 - val_accuracy: 0.6071 - val_loss: 0.9798 - learning_rate: 0.0010
Epoch 5/10
176/176 ━━━━━━━━━━━━━


✅ Exp 7: acc=0.7090 | loss=0.7935 | F1=0.7089 | 311.1s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_07.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_07.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_07_history.json

Experiment 8/16  |  {'num_conv_layers': 2, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'average'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 26s 132ms/step - accuracy: 0.3780 - loss: 1.4493 - val_accuracy: 0.5119 - val_loss: 1.1833 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 20s 113ms/step - accuracy: 0.5052 - loss: 1.1786 - val_accuracy: 0.5565 - val_loss: 1.1139 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 20s 116ms/step - accuracy: 0.5481 - loss: 1.1190 - val_accuracy: 0.5387 - val_loss: 1.0879 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 19s 109ms/step - accuracy: 0.5700 - loss: 1.0647 - val_accuracy: 0.6163 - val_loss: 0.9732 - learning_rate: 0.0010
Epoc


✅ Exp 8: acc=0.6680 | loss=0.8488 | F1=0.6644 | 205.8s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_08.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_08.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_08_history.json

Experiment 9/16  |  {'num_conv_layers': 3, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'max'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 30s 149ms/step - accuracy: 0.3902 - loss: 1.3806 - val_accuracy: 0.5155 - val_loss: 1.1269 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 95ms/step - accuracy: 0.5000 - loss: 1.1195 - val_accuracy: 0.5946 - val_loss: 0.9905 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 95ms/step - accuracy: 0.5817 - loss: 0.9843 - val_accuracy: 0.6291 - val_loss: 0.9172 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.6296 - loss: 0.9174 - val_accuracy: 0.6587 - val_loss: 0.8491 - learning_rate: 0.0010
Epoch 5/10
1


✅ Exp 9: acc=0.7490 | loss=0.6625 | F1=0.7444 | 195.4s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_09.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_09.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_09_history.json

Experiment 10/16  |  {'num_conv_layers': 3, 'filters': [32, 64], 'kernel_size': (3, 3), 'pooling_type': 'average'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 113ms/step - accuracy: 0.3748 - loss: 1.4345 - val_accuracy: 0.4549 - val_loss: 1.2645 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 103ms/step - accuracy: 0.4845 - loss: 1.2146 - val_accuracy: 0.5761 - val_loss: 1.0808 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 101ms/step - accuracy: 0.5463 - loss: 1.1078 - val_accuracy: 0.5771 - val_loss: 1.0643 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 20s 96ms/step - accuracy: 0.5857 - loss: 1.0313 - val_accuracy: 0.6295 - val_loss: 0.9296 - learning_rate: 0.0010
Epoch


✅ Exp 10: acc=0.6987 | loss=0.7716 | F1=0.6907 | 191.4s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_10.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_10.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_10_history.json

Experiment 11/16  |  {'num_conv_layers': 3, 'filters': [32, 64], 'kernel_size': (5, 5), 'pooling_type': 'max'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 123ms/step - accuracy: 0.3882 - loss: 1.3995 - val_accuracy: 0.4859 - val_loss: 1.1615 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.5090 - loss: 1.1312 - val_accuracy: 0.5287 - val_loss: 1.1242 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.6076 - loss: 0.9731 - val_accuracy: 0.5892 - val_loss: 0.9658 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 99ms/step - accuracy: 0.6462 - loss: 0.9019 - val_accuracy: 0.6758 - val_loss: 0.8281 - learning_rate: 0.0010
Epoch 5/1

...


✅ Exp 12: acc=0.7633 | loss=0.6227 | F1=0.7631 | 182.6s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_12.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_12.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_12_history.json

Experiment 13/16  |  {'num_conv_layers': 3, 'filters': [64, 128], 'kernel_size': (3, 3), 'pooling_type': 'max'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 33s 148ms/step - accuracy: 0.4018 - loss: 1.3605 - val_accuracy: 0.5194 - val_loss: 1.0976 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 22s 122ms/step - accuracy: 0.5283 - loss: 1.0825 - val_accuracy: 0.5803 - val_loss: 0.9730 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 20s 116ms/step - accuracy: 0.6059 - loss: 0.9655 - val_accuracy: 0.6598 - val_loss: 0.8858 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 22s 127ms/step - accuracy: 0.6519 - loss: 0.8769 - val_accuracy: 0.6779 - val_loss: 0.8087 - learning_rate: 0.0010
Epoch 


✅ Exp 13: acc=0.7557 | loss=0.6190 | F1=0.7584 | 264.2s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_13.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_13.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_13_history.json

Experiment 14/16  |  {'num_conv_layers': 3, 'filters': [64, 128], 'kernel_size': (3, 3), 'pooling_type': 'average'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 26s 125ms/step - accuracy: 0.3985 - loss: 1.3923 - val_accuracy: 0.5002 - val_loss: 1.1752 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5313 - loss: 1.1355 - val_accuracy: 0.5835 - val_loss: 1.0154 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 18s 102ms/step - accuracy: 0.5826 - loss: 1.0288 - val_accuracy: 0.6277 - val_loss: 0.9456 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.6190 - loss: 0.9441 - val_accuracy: 0.6762 - val_loss: 0.8515 - learning_rate: 0.0010
Epo


✅ Exp 14: acc=0.7477 | loss=0.6686 | F1=0.7414 | 186.3s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_14.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_14.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_14_history.json

Experiment 15/16  |  {'num_conv_layers': 3, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'max'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 46s 216ms/step - accuracy: 0.4166 - loss: 1.3115 - val_accuracy: 0.5119 - val_loss: 1.1822 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 160ms/step - accuracy: 0.5416 - loss: 1.0803 - val_accuracy: 0.5672 - val_loss: 1.0767 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 160ms/step - accuracy: 0.6102 - loss: 0.9612 - val_accuracy: 0.6177 - val_loss: 0.9302 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 28s 161ms/step - accuracy: 0.6366 - loss: 0.9047 - val_accuracy: 0.7260 - val_loss: 0.7646 - learning_rate: 0.0010
Epoch 


✅ Exp 15: acc=0.8190 | loss=0.4996 | F1=0.8198 | 306.1s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_15.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_15.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_15_history.json

Experiment 16/16  |  {'num_conv_layers': 3, 'filters': [64, 128], 'kernel_size': (5, 5), 'pooling_type': 'average'}

Epoch 1/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 31s 155ms/step - accuracy: 0.3941 - loss: 1.4132 - val_accuracy: 0.5237 - val_loss: 1.1507 - learning_rate: 0.0010
Epoch 2/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 38s 149ms/step - accuracy: 0.5154 - loss: 1.1590 - val_accuracy: 0.5290 - val_loss: 1.1187 - learning_rate: 0.0010
Epoch 3/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 25s 142ms/step - accuracy: 0.5728 - loss: 1.0516 - val_accuracy: 0.6056 - val_loss: 1.0090 - learning_rate: 0.0010
Epoch 4/10
176/176 ━━━━━━━━━━━━━━━━━━━━ 24s 134ms/step - accuracy: 0.5977 - loss: 1.0124 - val_accuracy: 0.6594 - val_loss: 0.9128 - learning_rate: 0.0010
Ep


✅ Exp 16: acc=0.7907 | loss=0.5904 | F1=0.7921 | 299.1s
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_16.keras
   💾 /content/drive/MyDrive/CNN_Experiments/models/model_16.h5
   💾 /content/drive/MyDrive/CNN_Experiments/results/model_16_history.json

✅ Semua eksperimen selesai!


In [ ]:
results_df = pd.DataFrame(results)
config_df  = pd.json_normalize(results_df['config'])
results_df = pd.concat([results_df.drop('config', axis=1), config_df], axis=1)

csv_path  = os.path.join(RESULT_DIR, 'experiment_results.csv')
json_path = os.path.join(RESULT_DIR, 'experiment_results.json')

results_df.to_csv(csv_path, index=False)
print(f"✅ CSV  : {csv_path}")

with open(json_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"✅ JSON : {json_path}")

results_df

✅ CSV  : /content/drive/MyDrive/CNN_Experiments/results/experiment_results.csv
✅ JSON : /content/drive/MyDrive/CNN_Experiments/results/experiment_results.json


,experiment_id,model_name,test_accuracy,test_loss,f1_macro,training_time_sec,num_params,best_epoch,best_val_loss,best_val_acc,num_conv_layers,filters,kernel_size,pooling_type
0,1,model_01,0.697333,0.787356,0.696523,212.410156,28486,10,0.793668,0.699679,2,"[32, 64]","[3, 3]",max
1,2,model_02,0.644333,0.938328,0.630996,176.817638,28486,10,0.921807,0.651585,2,"[32, 64]","[3, 3]",average
2,3,model_03,0.693000,0.809015,0.687355,194.768833,62790,10,0.819518,0.690417,2,"[32, 64]","[5, 5]",max
3,4,model_04,0.661333,0.885503,0.650550,189.589022,62790,10,0.880131,0.657642,2,"[32, 64]","[5, 5]",average
4,5,model_05,0.690667,0.806682,0.685535,208.380917,92934,10,0.816118,0.685429,2,"[64, 128]","[3, 3]",max
5,6,model_06,0.692000,0.806382,0.685957,185.218344,92934,10,0.805967,0.690773,2,"[64, 128]","[3, 3]",average
6,7,model_07,0.709000,0.793500,0.708868,311.071355,227078,10,0.799730,0.695404,2,"[64, 128]","[5, 5]",max
7,8,model_08,0.668000,0.848803,0.664371,205.774812,227078,10,0.834436,0.667260,2,"[64, 128]","[5, 5]",average
8,9,model_09,0.749000,0.662546,0.744373,195.434246,65414,10,0.684448,0.751336,3,"[32, 64]","[3, 3]",max
9,10,model_10,0.698667,0.771621,0.690707,191.431259,65414,10,0.783105,0.695404,3,"[32, 64]","[3, 3]",average
